# MNIST on Apple Neural Engine — KotlinGrad + KMP

End-to-end notebook orchestrating:
1. **Data prep** — download MNIST, convert to flat binary
2. **Native build** — compile ObjC ANE bridge + Kotlin/Native binary
3. **Training** — KotlinGrad scoped DSL → private ANE matmul dispatch
4. **Analysis** — loss curves, accuracy, ANE vs CPU timing breakdown

### Architecture
```
Kotlin DSL layer:   differentiable { (x matmul W1).relu() matmul W2 .crossEntropy(labels) }
                                     │                     │
GradScope (private): aneMatmul()──────┘                     └── aneMatmul()
                                     │                     │
Kotlin/Native cinterop ──────────────┘                     └─────────────────┘
                     │
ANEBridge.m (ObjC)  ──→  ane_runtime.h  ──→  AppleNeuralEngine.framework
```

### The KotlinGrad trick
The user writes pure symbolic math inside `differentiable { ... }`.
ANE hardware dispatch is **private** to `GradScope` — invisible to the DSL.
Weights are passed as *inputs* to ANE conv kernels (`mil_gen_matmul`),
so the kernels never need recompilation during training.

## Cell 1: Configuration

In [ ]:
import subprocess, os, sys, re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths (relative to notebook location in contrib/kmp-ane/)
PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"
MNIST_DIR = PROJECT_DIR / "mnist" / "data"

# Build targets
TRAINER_BIN_AUTOGRAD = "build/bin/macosArm64/mnistTrainerReleaseExecutable/mnist_trainer.kexe"
TRAINER_BIN_KOTLINGRAD = "build/bin/macosArm64/mnistKotlinGradReleaseExecutable/mnistKotlinGrad.kexe"

print(f"Project: {PROJECT_DIR}")
print(f"Data:    {DATA_DIR}")

## Cell 2: Download MNIST Data

In [ ]:
# Method A: flat binary for the Kotlin trainer (prepare_mnist.py)
if not (DATA_DIR / "mnist_train_images.bin").exists():
    print("Downloading MNIST (flat binary for Kotlin)...")
    !python3 prepare_mnist.py
else:
    print("✓ Flat binary MNIST data already present")

# Method B: IDX format for the standalone ObjC trainer
if not (MNIST_DIR / "train-images-idx3-ubyte").exists():
    print("Downloading MNIST (IDX format for ObjC demo)...")
    !cd mnist && python3 fetch_mnist.py
else:
    print("✓ IDX MNIST data already present")

# Quick sanity check
for name in ["mnist_train_images.bin", "mnist_train_labels.bin",
             "mnist_test_images.bin", "mnist_test_labels.bin"]:
    p = DATA_DIR / name
    print(f"  {name}: {p.stat().st_size:,} bytes")

## Cell 3: Visualize Some Digits

In [ ]:
# Load the flat binary data to show some samples
images = np.fromfile(DATA_DIR / "mnist_train_images.bin", dtype=np.float32).reshape(-1, 784)
labels = np.fromfile(DATA_DIR / "mnist_train_labels.bin", dtype=np.uint8)

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].reshape(28, 28), cmap="gray")
    ax.set_title(str(labels[i]), fontsize=10)
    ax.axis("off")
plt.suptitle("MNIST Training Samples", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Train: {len(images)} images, {len(np.unique(labels))} classes")
print(f"Pixel range: [{images.min():.2f}, {images.max():.2f}]")

## Cell 4: Build the Native Library + Kotlin Binary

In [ ]:
%%time

# Step 1: Compile the ObjC ANE bridge into a static library
print("=== Building ObjC ANE bridge ===")
r = subprocess.run(["make", "bridge"], capture_output=True, text=True, cwd=str(PROJECT_DIR))
if r.returncode != 0:
    print("STDERR:", r.stderr)
    raise RuntimeError("ObjC bridge build failed")
print(r.stdout[-200:] if len(r.stdout) > 200 else r.stdout)
print("✓ libanebridge.a built")

# Step 2: Build Kotlin/Native binaries
print("\n=== Building KotlinGrad MNIST trainer ===")
r = subprocess.run(
    ["./gradlew", "linkMnistKotlinGradReleaseExecutableMacosArm64", "--console=plain"],
    capture_output=True, text=True, cwd=str(PROJECT_DIR)
)
if r.returncode != 0:
    print("STDOUT:", r.stdout[-500:])
    print("STDERR:", r.stderr[-500:])
    raise RuntimeError("Gradle build failed")
print("✓ KotlinGrad binary built")

# Also build the Autograd trainer for comparison
print("\n=== Building Autograd MNIST trainer ===")
r2 = subprocess.run(
    ["./gradlew", "linkMnistTrainerReleaseExecutableMacosArm64", "--console=plain"],
    capture_output=True, text=True, cwd=str(PROJECT_DIR)
)
if r2.returncode == 0:
    print("✓ Autograd binary built")
else:
    print("⚠ Autograd build failed (non-critical)")

## Cell 5: Train — KotlinGrad DSL + ANE

This runs the KotlinGrad-style trainer:
- Forward matmuls dispatched **privately** to ANE inside `GradScope`
- Reverse-mode AD via `Var.backward()` (topological sort)
- Adam optimizer on `Param` objects

In [ ]:
%%time

def run_trainer(binary_path, label="trainer"):
    """Run a native trainer binary and parse CSV output."""
    bin_full = PROJECT_DIR / binary_path
    if not bin_full.exists():
        print(f"Binary not found: {bin_full}")
        return None, []
    
    print(f"Running {label}...")
    proc = subprocess.run(
        [str(bin_full)],
        capture_output=True, text=True,
        cwd=str(PROJECT_DIR),
        timeout=600  # 10 min max
    )
    
    stdout = proc.stdout
    print(stdout)
    
    # Parse CSV lines: epoch,step,loss,ane_ms,cpu_ms,test_acc
    rows = []
    for line in stdout.split("\n"):
        line = line.strip()
        if line.startswith("#") or not line:
            continue
        parts = line.split(",")
        if len(parts) >= 5:
            try:
                epoch = int(parts[0])
                step = int(parts[1])
                loss = float(parts[2])
                ane_ms = float(parts[3])
                cpu_ms = float(parts[4])
                test_acc = float(parts[5]) if len(parts) > 5 and parts[5].strip() else None
                rows.append({
                    "epoch": epoch, "step": step, "loss": loss,
                    "ane_ms": ane_ms, "cpu_ms": cpu_ms, "test_acc": test_acc
                })
            except (ValueError, IndexError):
                continue
    
    return stdout, rows

# Run KotlinGrad trainer
kg_stdout, kg_rows = run_trainer(TRAINER_BIN_KOTLINGRAD, "KotlinGrad + ANE")

## Cell 6: Training Curves

In [ ]:
if kg_rows:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Loss curve ---
    steps = [r["epoch"] + r["step"] / max(r2["step"] for r2 in kg_rows if r2["epoch"] == r["epoch"])
             for r in kg_rows]
    losses = [r["loss"] for r in kg_rows]
    ax1.plot(steps, losses, 'b-', alpha=0.7, linewidth=1.5)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Cross-Entropy Loss")
    ax1.set_title("Training Loss (KotlinGrad → ANE)")
    ax1.grid(True, alpha=0.3)
    
    # --- Test accuracy at epoch boundaries ---
    epoch_rows = [r for r in kg_rows if r["test_acc"] is not None]
    if epoch_rows:
        epochs = [r["epoch"] for r in epoch_rows]
        accs = [r["test_acc"] for r in epoch_rows]
        ax2.plot(epochs, accs, 'go-', markersize=8, linewidth=2)
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Test Accuracy (%)")
        ax2.set_title("Test Accuracy")
        ax2.set_ylim(0, 100)
        ax2.grid(True, alpha=0.3)
        for e, a in zip(epochs, accs):
            ax2.annotate(f"{a:.1f}%", (e, a), textcoords="offset points",
                        xytext=(0, 10), ha='center', fontsize=9)
    
    plt.suptitle("MNIST — KotlinGrad DSL + Apple Neural Engine", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No training data to plot (build or run failed)")

## Cell 7: ANE vs CPU Timing Breakdown

In [ ]:
if kg_rows:
    # Take the last row (total accumulated times)
    last = kg_rows[-1]
    ane_ms = last["ane_ms"]
    cpu_ms = last["cpu_ms"]
    total = ane_ms + cpu_ms
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart
    sizes = [ane_ms, cpu_ms]
    labels_pie = [f"ANE forward\n{ane_ms:.0f}ms", f"CPU (back+rest)\n{cpu_ms:.0f}ms"]
    colors = ['#4CAF50', '#2196F3']
    ax1.pie(sizes, labels=labels_pie, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
    ax1.set_title("Compute Time Distribution")
    
    # Per-epoch timing (if we have epoch-end rows)
    epoch_rows = [r for r in kg_rows if r["test_acc"] is not None]
    if len(epoch_rows) > 1:
        # Compute per-epoch deltas
        ane_per_epoch = []
        cpu_per_epoch = []
        prev_ane = 0; prev_cpu = 0
        for r in epoch_rows:
            ane_per_epoch.append(r["ane_ms"] - prev_ane)
            cpu_per_epoch.append(r["cpu_ms"] - prev_cpu)
            prev_ane = r["ane_ms"]
            prev_cpu = r["cpu_ms"]
        
        epochs = [r["epoch"] for r in epoch_rows]
        x = np.arange(len(epochs))
        w = 0.35
        ax2.bar(x - w/2, ane_per_epoch, w, label="ANE forward", color='#4CAF50')
        ax2.bar(x + w/2, cpu_per_epoch, w, label="CPU backward", color='#2196F3')
        ax2.set_xticks(x)
        ax2.set_xticklabels([str(e) for e in epochs])
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Time (ms)")
        ax2.set_title("Per-Epoch Compute Time")
        ax2.legend()
        ax2.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle("ANE Training Time Breakdown", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    if total > 0:
        print(f"\nANE utilization: {ane_ms/total*100:.1f}%")
        print(f"Total compute:   {total:.0f} ms ({total/1000:.1f}s)")
else:
    print("No timing data available")

## Cell 8: Architecture Recap

### KotlinGrad DSL (what the user writes)

```kotlin
// Pure symbolic math — no hardware references
val loss = scope.forward {
    val x = input(images, "images")
    val h = (x matmul param(W1)).addBias(param(b1)).relu()
    val logits = (h matmul param(W2)).addBias(param(b2))
    logits.crossEntropy(labels)
}
loss.backward()  // reverse-mode AD
optimizer.step()  // Adam on Param.grad
```

### What happens under the hood

```
KotlinGrad Var.matmul()     ← public DSL operator
  │
  └─→ GradScope.aneMatmul() ← PRIVATE, invisible to user
        │
        ├─→ aneCompileConv()  ← cached, weights-as-input (0 recompiles)
        │     └─→ ANEDevice.compileConv()  [cinterop]
        │           └─→ ane_bridge_compile_conv()  [C]
        │                 └─→ _ANEInMemoryModel  [private framework]
        │
        └─→ ANEDevice.convEval()  [cinterop]
              └─→ ane_bridge_conv_eval()  [C]
                    └─→ IOSurface → ANE → IOSurface
```

### Key insight: zero recompilation

Unlike `mil_gen_conv` (which bakes weights into the MIL program),
`mil_gen_matmul` treats weights as **input tensors**. This means:

- Only **2 ANE kernels** compiled for the entire 784→128→10 network
- Weights updated every step by writing new values to IOSurface
- Zero recompilation overhead during training

## Cell 9: Compare Autograd vs KotlinGrad (Optional)

Run both the old Autograd tape-based trainer and the new KotlinGrad scoped
DSL trainer side by side.

In [ ]:
# Run autograd trainer if available
autograd_bin = PROJECT_DIR / TRAINER_BIN_AUTOGRAD
if autograd_bin.exists():
    print("Running Autograd + ANE trainer...")
    ag_proc = subprocess.run(
        [str(autograd_bin)],
        capture_output=True, text=True,
        cwd=str(PROJECT_DIR),
        timeout=600
    )
    print(ag_proc.stdout[-1000:])
else:
    print(f"Autograd binary not found at {autograd_bin}")
    print("Build with: ./gradlew linkMnistTrainerReleaseExecutableMacosArm64")

## Cell 10: Full Stack Summary

| Layer | File | Language | Purpose |
|-------|------|----------|---------|
| KotlinGrad DSL | `KotlinGrad.kt` | Kotlin | Scoped AD: `differentiable { }`, `Var.backward()` |
| Param / Tensor | `Tensor.kt` | Kotlin | Weight storage, Kaiming init |
| ANEDevice | `ANEDeviceNative.kt` | Kotlin/Native | cinterop → C bridge |
| ANE Bridge | `ANEBridge.m` | ObjC | IOSurface I/O, MIL gen, Apple private API |
| MNIST Trainer | `MnistKotlinGrad.kt` | Kotlin/Native | Data loading, train loop, metrics |
| Notebook | `notebook.ipynb` | Python | Orchestration, visualization |